# Agent Scouting Workbench

This notebook is a chat-first front end for the modular multi-agent engine in `engine/`.

Modes:
- Structured report: choose a recruit and target school.
- Open chat: ask scouting questions and continue with follow-ups.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, display


def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'data').exists() and (candidate / 'engine').exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from engine.graph import get_scout_graph
from engine.state import initial_chat_state, initial_structured_state

graph = get_scout_graph()
print(f'Graph ready. Project root: {PROJECT_ROOT}')

In [ ]:
recruits_path = PROJECT_ROOT / 'data' / 'modeling_datasets' / 'recruits' / 'master_recruits_2015_2028.csv'
df_recruits = pd.read_csv(recruits_path) if recruits_path.exists() else pd.DataFrame()

if not df_recruits.empty:
    df_recruits['player_label'] = (
        df_recruits['player_name'].astype(str) + ' | ' +
        df_recruits['position'].astype(str) + ' | ' +
        df_recruits['high_school'].astype(str) + ' | ' +
        df_recruits['year'].astype(str)
    )
    player_options = df_recruits['player_label'].head(1000).tolist()
    label_to_id = dict(zip(df_recruits['player_label'], df_recruits['recruit_id'].astype(str)))
else:
    player_options = []
    label_to_id = {}

TARGET_TEAMS = [
    'Alabama', 'Auburn', 'Clemson', 'Colorado', 'Duke', 'Florida', 'Florida State',
    'Georgia', 'Georgia Tech', 'LSU', 'Miami', 'Michigan', 'NC State', 'Notre Dame',
    'Ohio State', 'Ole Miss', 'Oregon', 'South Carolina', 'Tennessee', 'Texas',
    'Texas A&M', 'Charlotte', 'USC', 'Virginia Tech', 'Wake Forest'
]
print(f'Loaded recruits: {len(player_options)}')

In [ ]:
year_dropdown = widgets.Dropdown(options=[2026, 2027, 2028], value=2026, description='Year:')
player_dropdown = widgets.Combobox(
    options=player_options,
    placeholder='Select player',
    description='Player:',
    ensure_option=False,
    layout=widgets.Layout(width='760px'),
)
team_dropdown = widgets.Dropdown(options=TARGET_TEAMS, value='Wake Forest', description='Target Team:')
generate_button = widgets.Button(description='Generate Report', button_style='primary')

chat_input = widgets.Textarea(
    placeholder='Ask a scouting question...',
    description='Chat:',
    layout=widgets.Layout(width='760px', height='100px'),
)
chat_button = widgets.Button(description='Send Chat', button_style='info')

status = widgets.HTML(value='')
output = widgets.Output()

chat_memory = []
last_structured_state = None


def run_structured(_):
    global last_structured_state
    output.clear_output()

    label = str(player_dropdown.value).strip()
    recruit_id = label_to_id.get(label, '')
    if not recruit_id:
        status.value = "<span style='color:#b91c1c;font-weight:600;'>Choose a valid player from the list.</span>"
        return

    state = initial_structured_state(
        player_name=label.split('|')[0].strip(),
        recruit_id=recruit_id,
        target_team=str(team_dropdown.value),
        year=int(year_dropdown.value),
    )
    state['user_query'] = 'Generate a full scouting report with current context and recommendation.'

    status.value = "<span style='color:#1d4ed8;font-weight:600;'>Running multi-agent pipeline...</span>"
    final_state = graph.invoke(state)
    last_structured_state = final_state

    with output:
        display(Markdown(f"## Structured Report\n\n{final_state.get('final_report', '')}"))
        if final_state.get('errors'):
            err_lines = '\n'.join([f"- {e}" for e in final_state['errors']])
            display(Markdown(f"### Errors\n{err_lines}"))

    status.value = "<span style='color:#166534;font-weight:600;'>Structured report complete.</span>"


def run_chat(_):
    global chat_memory, last_structured_state
    output.clear_output()

    query = str(chat_input.value).strip()
    if not query:
        status.value = "<span style='color:#b91c1c;font-weight:600;'>Enter a chat question first.</span>"
        return

    if last_structured_state:
        state = dict(last_structured_state)
        state['mode'] = 'chat'
        state['user_query'] = query
        state['conversation_history'] = chat_memory
    else:
        state = initial_chat_state(query)
        state['conversation_history'] = chat_memory

    status.value = "<span style='color:#1d4ed8;font-weight:600;'>Running chat agents...</span>"
    final_state = graph.invoke(state)
    chat_memory = final_state.get('conversation_history', chat_memory)

    with output:
        display(Markdown(f"## Chat Response\n\n{final_state.get('final_report', '')}"))
        if final_state.get('citations'):
            display(Markdown('### Citations'))
            for c in final_state['citations'][-8:]:
                src = c.get('source_name', 'source')
                url = c.get('source_url', '')
                line = f"- {src}" if not url else f"- [{src}]({url})"
                display(Markdown(line))

    status.value = "<span style='color:#166534;font-weight:600;'>Chat response complete.</span>"


generate_button.on_click(run_structured)
chat_button.on_click(run_chat)

display(
    widgets.VBox(
        [
            widgets.HTML('<h3>Structured Report Mode</h3>'),
            widgets.HBox([year_dropdown, team_dropdown]),
            player_dropdown,
            generate_button,
            widgets.HTML('<h3>Open Chat Mode</h3>'),
            chat_input,
            chat_button,
            status,
            output,
        ]
    )
)